# Capstone — The Search Intelligence ML Pipeline

## 1. Abstract
Identifying which digital content pieces are actively decaying in search performance is a resource-intensive manual process for content teams. This research developed a machine learning pipeline using Random Forest classifiers to predict content performance degradation based on 90-day search visibility metrics. By analyzing real production search data (79 million rows reduced to anonymized client sets), the model achieved an honest cross-validation accuracy of 57.65% on unseen clients. The output is a ranked action playbook that prioritizes specific articles for editorial review, mathematically optimizing content refresh strategies.

## 2. Introduction & Problem Statement
At FlyRank, a key problem Content Strategists face is deciding *where* to spend their limited writing hours. Not all content decays equally, and looking at basic traffic drops often misses the underlying causes (e.g., impression drops vs. CTR decay). We aimed to build a decision-support model that flags articles likely to experience a downward trend, tying our findings directly to the real FlyRank content problem: maximizing the ROI of weekly editorial updates by prioritizing content that shows statistical signals of decay.

## 3. Data
**Source:** `content_refresh_anonymized.csv` (FlyRank ML Internship Release).

**Window:** 90-day historical search performance metrics.

**Exclusions:** 
- Rows with missing `trend_direction` labels were dropped because they represent ambiguous or unclassified states, making them invalid for supervised training.
- Articles with `days_since_last_update` < 30 were excluded from the final action playbook because newer content hasn't had enough time to establish a reliable baseline trend.

## 4. Methodology
- **Label Definition:** A binary classification target `is_declining`, derived from `trend_direction == 'down'`.
- **Features:** `days_since_last_update`, `ctr`, `impressions_90d`, `clicks_90d`, `avg_position`, `engagement_rate`, `word_count`, `has_word_count`.
- **Assumptions:** We assumed that a lack of a `word_count` value could be structurally significant (e.g., dynamic pages), so we imputed it with 0 but added a `has_word_count` binary feature to preserve that signal.
- **Baseline:** A Random Forest Classifier trained with standard 80/20 random splits.
- **Validation Design & Leakage Checks:** We discovered severe client-level data leakage (accuracy 66.58%) when using random splits. We fixed this by implementing `GroupShuffleSplit` on `client_id`, ensuring the model evaluates truly unseen clients (Honest Accuracy 57.65%).

## 5. Results (vs baseline)
When evaluated properly against unseen clients, the model's accuracy drops to realistic levels, demonstrating that the initial high score was an illusion of leakage.

| Evaluation Method | Accuracy | Precision | Recall | Leakage Status |
| :--- | :--- | :--- | :--- |
| Naive Split (Random 80/20) | 66.58% | 0.65 | 0.68 | Leaking Client Data |
| Honest Split (GroupShuffle) | 57.65% | 0.55 | 0.61 | Zero Client Leakage |

## 6. Limitations
- This model does not understand the *semantic meaning* or subjective quality of the content. It only sees behavioral metrics.
- The model will incorrectly flag evergreen/static pages (like Privacy Policies) as "declining" due to low engagement. 
- The outputs are strictly **directional decision-support** tools. They should never be used to automatically delete content or redirect URLs without human editorial sign-off.

## 7. Ranked Recommendations (The Playbook)
Using a probability threshold of > 65%, the model outputs an `action_queue`. Here is a sample of the generated reason codes used by the editorial team:

1. **Stale & Low CTR - Needs Content Update** (e.g., > 180 days old, CTR < 2.0%)
2. **High Visibility, Poor CTR - Needs Better Title/Meta** (e.g., > 1000 impressions, CTR < 1.0%)
3. **General Decay - Review Content Quality**

## 8. Reproducibility
The complete codebase, data processing notebooks, leakage audits, and action playbook scripts can be inspected in the main repository: 
[https://github.com/nnichaelangello/FlyRank.AI](https://github.com/nnichaelangello/FlyRank.AI)

## 9. Acknowledgments
Built on the FlyRank ML Internship dataset. Data provided by [https://flyrank.ai](https://flyrank.ai).

---
## Appendix: ML-12 Tell the Story Deliverables

### 5-Minute Demo Outline
1. **The Question (1 min):** "How can we stop content strategists from guessing which articles are decaying and wasting hours reviewing the wrong pages?"
2. **The Method (1 min):** "I built a Random Forest classifier to predict content decay using 90-day search signals. I specifically used GroupShuffleSplit to prevent the model from memorizing client-specific behaviors."
3. **One Chart (1 min):** *Show the `queue_distribution.png` histogram.* "This shows how we threshold the probabilities. Only the top 65%+ risk articles make it to the editor's desk."
4. **One Honest Result (1 min):** "My honest accuracy is 57.65%. While modest, it's a true baseline on unseen clients, whereas a naive split falsely claimed 66% due to data leakage."
5. **One Recommendation (1 min):** "If an article is over 6 months old and has a CTR under 2%, the script flags it with a specific 'Needs Content Update' reason code, giving the human editor immediate context."

### Social Post Cut
Just wrapped up a Machine Learning capstone analyzing 79 million rows of production search data! 🚀 I built a Random Forest classifier to predict content decay, but the real win was catching a massive data leak. By switching from a naive random split to a GroupShuffleSplit based on client IDs, I sacrificed "vanity metrics" (66% accuracy) for an honest, leak-free baseline (57%). It’s a great reminder that how you validate your model is just as important as how you build it.

### 3-Sentence Employer Summary
I engineered a machine learning pipeline using Random Forest to predict search content degradation, trained on an anonymized dataset derived from 79 million production search records. By implementing rigorous validation techniques (GroupShuffleSplit) to eliminate client-level data leakage, I established an honest baseline model. I then operationalized the model's outputs into a ranked action playbook, directly translating probability metrics into business-ready editorial recommendations.